In [17]:
import os
import re
import zipfile
import textwrap
import pandas as pd

from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.platypus import Table, TableStyle, Paragraph
from reportlab.lib.units import inch
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from PIL import Image

In [ ]:
# Caminho base dos arquivos recebidos
caminho_tabelas = ('/home/gaolgo/documents/repositories/murta_engenharia/pdfs_sismepe/tabelas_recebidas/')

mes = 2
ano = "2026"

In [19]:
MESES_PT = {
    1: "JANEIRO",
    2: "FEVEREIRO",
    3: "MARÇO",
    4: "ABRIL",
    5: "MAIO",
    6: "JUNHO",
    7: "JULHO",
    8: "AGOSTO",
    9: "SETEMBRO",
    10: "OUTUBRO",
    11: "NOVEMBRO",
    12: "DEZEMBRO",
}

LINHAS_POR_PAGINA = 20
TAMANHO_QUEBRA_TEXTO = 20
LIMITE_TEXTO_PEQUENO = 30


def sanitizar_nome_arquivo(nome):
    """Remove caracteres inválidos para nome de arquivo."""
    return re.sub(r'[\\/:"*?<>|]+', "_", str(nome))


def formatar_moeda_brl(valor):
    """Formata um valor numérico no padrão monetário brasileiro."""
    if pd.isnull(valor):
        valor = 0

    if isinstance(valor, (int, float)):
        return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

    return valor


def formatar_colunas_monetarias(df, colunas_monetarias):
    """Formata as colunas monetárias de um dataframe."""
    df = df.copy()

    for coluna in colunas_monetarias:
        if coluna in df.columns:
            df[coluna] = df[coluna].fillna(0).apply(formatar_moeda_brl)

    return df


def criar_estilos_paragrafo():
    """Cria os estilos de parágrafo usados nas células da tabela."""
    estilos = getSampleStyleSheet()

    estilo_normal = ParagraphStyle(
        name="EstiloNormal",
        parent=estilos["Normal"],
        fontName="Helvetica",
        fontSize=8,
        leading=10,
    )

    estilo_pequeno = ParagraphStyle(
        name="EstiloPequeno",
        parent=estilos["Normal"],
        fontName="Helvetica",
        fontSize=6,
        leading=8,
    )

    return estilo_normal, estilo_pequeno


def formatar_texto_celula(valor, estilo_normal, estilo_pequeno, quebra_texto=20, limite_texto=30):
    """Converte um valor em parágrafo com quebra de linha automática."""
    if pd.isnull(valor):
        valor = 0

    texto = str(valor)
    texto_quebrado = "<br/>".join(textwrap.wrap(texto, quebra_texto))
    estilo = estilo_pequeno if len(texto) > limite_texto else estilo_normal

    return Paragraph(texto_quebrado, estilo)


def quebrar_texto_dataframe(df, quebra_texto=20, limite_texto=30):
    """Aplica quebra de texto em todas as células do dataframe."""
    df = df.copy()
    estilo_normal, estilo_pequeno = criar_estilos_paragrafo()

    for coluna in df.columns:
        df[coluna] = df[coluna].apply(
            lambda valor: formatar_texto_celula(
                valor=valor,
                estilo_normal=estilo_normal,
                estilo_pequeno=estilo_pequeno,
                quebra_texto=quebra_texto,
                limite_texto=limite_texto,
            )
        )

    return df


def preparar_dados_tabela(df):
    """Converte o dataframe para o formato esperado pela tabela do ReportLab."""
    return [df.columns.to_list()] + df.values.tolist()


def obter_dimensoes_logo(caminho_logo, largura_alvo=inch * 1.5):
    """Calcula a largura e altura proporcionais da logo."""
    imagem_logo = Image.open(caminho_logo)
    largura_logo, altura_logo = imagem_logo.size
    proporcao = largura_logo / altura_logo

    nova_largura = largura_alvo
    nova_altura = nova_largura / proporcao

    return nova_largura, nova_altura


def desenhar_cabecalho_pdf(c, largura_pagina, altura_pagina, caminho_logo, mes, ano, rotulo_titulo, nome_grupo, pagina_atual, total_paginas):
    """Desenha o cabeçalho padrão de cada página do PDF."""
    largura_logo, altura_logo = obter_dimensoes_logo(caminho_logo)

    posicao_x_logo = largura_pagina - largura_logo - inch * 0.25
    posicao_y_logo = altura_pagina - altura_logo - inch * 0.25
    c.drawImage(caminho_logo, posicao_x_logo, posicao_y_logo, largura_logo, altura_logo)

    posicao_x_texto = inch * 0.25
    c.setFont("Helvetica-Bold", 10)
    c.drawString(posicao_x_texto, altura_pagina - inch, "Relatório gerencial")
    c.drawString(posicao_x_texto, altura_pagina - 1.2 * inch, "Contrato: Polícia Militar do Estado de Pernambuco")
    c.drawString(posicao_x_texto, altura_pagina - 1.4 * inch, f"Período de apuração: {MESES_PT[mes]}/{ano}")
    c.drawString(posicao_x_texto, altura_pagina - 1.6 * inch, f"{rotulo_titulo}: {nome_grupo}")

    c.setFont("Helvetica", 8)
    c.drawRightString(largura_pagina - inch, inch / 2, f"Página {pagina_atual} de {total_paginas}")


def criar_tabela(dados_tabela, largura_pagina):
    """Cria uma tabela estilizada para o PDF."""
    quantidade_colunas = len(dados_tabela[0])
    larguras_colunas = [largura_pagina / quantidade_colunas] * quantidade_colunas

    tabela = Table(dados_tabela, colWidths=larguras_colunas)
    tabela.setStyle(
        TableStyle(
            [
                ("BACKGROUND", (0, 0), (-1, 0), colors.grey),
                ("TEXTCOLOR", (0, 0), (-1, 0), colors.whitesmoke),
                ("ALIGN", (0, 0), (-1, -1), "CENTER"),
                ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                ("BOTTOMPADDING", (0, 0), (-1, 0), 10),
                ("BACKGROUND", (0, 1), (-1, -1), colors.beige),
                ("GRID", (0, 0), (-1, -1), 0.5, colors.lightgrey),
            ]
        )
    )

    return tabela


def desenhar_tabela_no_canvas(c, tabela, largura_pagina, altura_pagina):
    """Posiciona e desenha a tabela na página do PDF."""
    largura_tabela, altura_tabela = tabela.wrap(0, 0)
    posicao_x = (largura_pagina - largura_tabela) / 2
    posicao_y = altura_pagina - 2 * inch - altura_tabela

    tabela.wrapOn(c, posicao_x, posicao_y)
    tabela.drawOn(c, posicao_x, posicao_y)


def adicionar_tabela_com_paginacao(c, df, largura_pagina, altura_pagina, caminho_logo, mes, ano, rotulo_titulo, nome_grupo):
    """Adiciona uma tabela paginada ao PDF."""
    total_linhas = len(df)
    total_paginas = (total_linhas // LINHAS_POR_PAGINA) + (1 if total_linhas % LINHAS_POR_PAGINA else 0)

    for indice_pagina in range(total_paginas):
        pagina_atual = indice_pagina + 1

        desenhar_cabecalho_pdf(
            c=c,
            largura_pagina=largura_pagina,
            altura_pagina=altura_pagina,
            caminho_logo=caminho_logo,
            mes=mes,
            ano=ano,
            rotulo_titulo=rotulo_titulo,
            nome_grupo=nome_grupo,
            pagina_atual=pagina_atual,
            total_paginas=total_paginas,
        )

        linha_inicial = indice_pagina * LINHAS_POR_PAGINA
        linha_final = min(linha_inicial + LINHAS_POR_PAGINA, total_linhas)

        df_pagina = df.iloc[linha_inicial:linha_final]
        dados_tabela = preparar_dados_tabela(df_pagina)
        tabela = criar_tabela(dados_tabela, largura_pagina)
        desenhar_tabela_no_canvas(c, tabela, largura_pagina, altura_pagina)

        if pagina_atual < total_paginas:
            c.showPage()


def preparar_dataframe_grupo(dados, coluna_agrupadora, colunas_monetarias):
    """Prepara os dados de um grupo para exportação em PDF."""
    dados = dados.drop(columns=[coluna_agrupadora]).fillna(0)
    dados = formatar_colunas_monetarias(dados, colunas_monetarias)
    dados = quebrar_texto_dataframe(
        dados,
        quebra_texto=TAMANHO_QUEBRA_TEXTO,
        limite_texto=LIMITE_TEXTO_PEQUENO,
    )
    return dados


def montar_nome_pdf(pasta_saida, prefixo, grupo, rotulo_titulo, mes, ano):
    """Monta o nome final do arquivo PDF."""
    grupo_limpo = sanitizar_nome_arquivo(grupo)
    titulo_limpo = sanitizar_nome_arquivo(rotulo_titulo)

    return os.path.join(
        pasta_saida,
        f"{prefixo}_{grupo_limpo}_por_{titulo_limpo}_{MESES_PT[mes].lower()}_{ano}.pdf",
    )


def compactar_pdfs(lista_pdfs, pasta_saida, prefixo, ano, mes):
    """Compacta todos os PDFs gerados em um arquivo ZIP."""
    nome_zip = os.path.join(pasta_saida, f"{prefixo}_{ano}_{mes:02}.zip")

    with zipfile.ZipFile(nome_zip, "w") as zipf:
        for arquivo_pdf in lista_pdfs:
            zipf.write(arquivo_pdf, os.path.basename(arquivo_pdf))

    print(f"Arquivo ZIP salvo em: {nome_zip}")


def gerar_pdfs(
    dataframe,
    colunas_monetarias,
    pasta_saida,
    rotulo_titulo,
    coluna_agrupadora,
    mes,
    ano,
    prefixo,
    caminho_logo="tabelas_recebidas/logo_murta.png",
):
    """Gera PDFs por grupo e cria um ZIP com todos os arquivos."""
    os.makedirs(pasta_saida, exist_ok=True)
    lista_pdfs = []

    for grupo, dados in dataframe.groupby(coluna_agrupadora):
        nome_pdf = montar_nome_pdf(
            pasta_saida=pasta_saida,
            prefixo=prefixo,
            grupo=grupo,
            rotulo_titulo=rotulo_titulo,
            mes=mes,
            ano=ano,
        )
        lista_pdfs.append(nome_pdf)

        dados_preparados = preparar_dataframe_grupo(
            dados=dados,
            coluna_agrupadora=coluna_agrupadora,
            colunas_monetarias=colunas_monetarias,
        )

        c = canvas.Canvas(nome_pdf, pagesize=A4)
        largura_pagina, altura_pagina = A4

        adicionar_tabela_com_paginacao(
            c=c,
            df=dados_preparados,
            largura_pagina=largura_pagina,
            altura_pagina=altura_pagina,
            caminho_logo=caminho_logo,
            mes=mes,
            ano=ano,
            rotulo_titulo=rotulo_titulo,
            nome_grupo=grupo,
        )

        c.save()

    compactar_pdfs(
        lista_pdfs=lista_pdfs,
        pasta_saida=pasta_saida,
        prefixo=prefixo,
        ano=ano,
        mes=mes,
    )


def processar_dados_procedimentos(df, dicionario_selecao_renomeacao, quantidade_colunas_grupo=2):
    """Seleciona, renomeia, agrupa e calcula métricas dos procedimentos."""
    df_selecionado = df[list(dicionario_selecao_renomeacao.keys())].copy()
    df_selecionado = df_selecionado.rename(columns=dicionario_selecao_renomeacao)

    colunas_grupo = list(dicionario_selecao_renomeacao.values())[:quantidade_colunas_grupo]

    df_agrupado = (
        df_selecionado
        .groupby(colunas_grupo, as_index=False)
        .sum()
    )

    df_agrupado["Custo Médio Eventos"] = (
        df_agrupado["Custo Assistencial"]
        .div(df_agrupado["Qtde Eventos Assistenciais"].replace(0, pd.NA))
        .round(2)
    )

    df_agrupado["Qtde Eventos Assistenciais"] = (
        df_agrupado["Qtde Eventos Assistenciais"]
        .fillna(0)
        .astype(int)
    )

    df_agrupado = df_agrupado.sort_values("Custo Assistencial", ascending=False)

    return df_agrupado

In [ ]:
# Nomes dos arquivos
arquivo_autorizacoes = f"SISMEPE_Autorizacoes_20260{mes}.xlsx"
arquivo_contas = f"SISMEPE_ContasMedicas_{ano}0{mes - 1}.xlsx"
arquivo_procedimentos = f"SISMEPE_Procedimentos_{ano}0{mes - 1}.xlsx"

# Caminhos completos
caminho_autorizacoes = caminho_tabelas + arquivo_autorizacoes
caminho_contas = caminho_tabelas + arquivo_contas
caminho_procedimentos = caminho_tabelas + arquivo_procedimentos

# Leitura dos arquivos
autorizacoes = pd.read_excel(caminho_autorizacoes)
contas = pd.read_excel(caminho_contas)
procedimentos = pd.read_excel(caminho_procedimentos)

# Colunas monetárias padrão dos relatórios
colunas_monetarias = ["Custo Assistencial", "Custo Médio Eventos"]


# =========================================================
# 1. RELATÓRIO DE PROCEDIMENTOS POR CLASSE
# =========================================================
mapeamento_colunas_classe_procedimento = {
    "Classificacao": "Classe",
    "DESCRICAO": "Procedimento",
    "QTD_PAGO": "Qtde Eventos Assistenciais",
    "VLR_PAGO": "Custo Assistencial"}

procedimentos_agrupados_classe = processar_dados_procedimentos(
    df=procedimentos,
    dicionario_selecao_renomeacao=mapeamento_colunas_classe_procedimento)

gerar_pdfs(
    dataframe=procedimentos_agrupados_classe,
    colunas_monetarias=colunas_monetarias,
    pasta_saida="pdfs_procedimentos",
    rotulo_titulo="Custo do procedimento",
    coluna_agrupadora="Classe",
    mes=mes,
    ano=ano,
    prefixo="classes")


# =========================================================
# 2. RELATÓRIO DE PRESTADORES POR CLASSE
# =========================================================
mapeamento_colunas_classe_prestador = {
    "Classificacao": "Classe",
    "Prestador": "Prestador",
    "QTD_PAGO": "Qtde Eventos Assistenciais",
    "VLR_PAGO": "Custo Assistencial",}

procedimentos_agrupados_prestador = processar_dados_procedimentos(
    df=procedimentos,
    dicionario_selecao_renomeacao=mapeamento_colunas_classe_prestador,)

gerar_pdfs(
    dataframe=procedimentos_agrupados_prestador,
    colunas_monetarias=colunas_monetarias,
    pasta_saida="pdfs_procedimentos_prestadores",
    rotulo_titulo="Custo do procedimento",
    coluna_agrupadora="Classe",
    mes=mes,
    ano=ano,
    prefixo="prestadores",)

Arquivo ZIP salvo em: pdfs_procedimentos/classes_2026_03.zip
Arquivo ZIP salvo em: pdfs_procedimentos_prestadores/prestadores_2026_03.zip
